# Examine De Machabeis Corpus

This notebook evaluates the hold-out work "X3_machabeis" against the shared hexameter test corpus to assess its linguistic compatibility with **H** (Hildebert) and with the other comparison profiles.

## Approach

We use stylometric analysis (Lambda-G) on 100 random ~750-token samples from *De Machabeis*. The shared logistic regression model calibrated on the known-authorship hexameter training corpus converts raw LambdaG scores into calibrated probabilities.

*Certamina* (`X4_certamina`) remains fully excluded from training, test splitting, and calibration. It is reintroduced only at the application stage as an **analysis-only candidate**, so that it can appear in the same comparative ranking without entering the reference pool for the other candidates.

## Stages

1. **Setup** — Load the shared test corpus, the two hold-out corpora, and the trained model
2. **Evaluation vs H** — Run 100 parallel comparisons with Hildebert
3. **Comparative evaluation** — Run 100 paired comparisons against each background candidate and against *Certamina* as an analysis-only candidate. Within each replicate, every candidate is evaluated on exactly the same random sample from *De Machabeis*.
4. **Visualization** — Inspect raw scores, calibrated probabilities, and the comparative ranking


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import joblib
import ray
from tqdm import tqdm
from typing import Any

from sentence_corpora import TwoLevelCorpus
from sentence_corpora.lambdag.av import LambdaGAV

import mpl_fontkit as fk

# Set up Roboto Condensed font
fk.install("Roboto Condensed")

# Set seaborn theme
sns.set_theme(
    context="paper",
    font="Roboto Condensed",
    style="white",
    rc={
        "figure.figsize": (10, 4),
        "figure.dpi": 300,
        "axes.facecolor": "white",
        "figure.facecolor": "white",
        "axes.unicode_minus": False,
        "font.size": 11,
        "axes.titlesize": 12,
        "axes.labelsize": 10,
    },
)

In [ ]:
# ---------------------------------------------------------
# Common files produced by the general calibration pipeline
# ---------------------------------------------------------

GENERAL_DIR = Path("..").resolve()

TEST_CORPUS_PATH = GENERAL_DIR / "test_corpus.pkl"
MACHABEIS_CORPUS_PATH = GENERAL_DIR / "machabeis_corpus.pkl"
CERTAMINA_CORPUS_PATH = GENERAL_DIR / "certamina_corpus.pkl"
MODEL_PATH = GENERAL_DIR / "logistic_regression_model_uncorrected.pkl"

# Check that the required general files exist
for path in [
    TEST_CORPUS_PATH,
    MACHABEIS_CORPUS_PATH,
    CERTAMINA_CORPUS_PATH,
    MODEL_PATH,
]:
    if not path.is_file():
        raise FileNotFoundError(f"Required file not found: {path}")

print(f"General analysis directory: {GENERAL_DIR}")

# Load corpora
test_corpus = TwoLevelCorpus.from_pickle(
    str(TEST_CORPUS_PATH)
)

machabeis_corpus = TwoLevelCorpus.from_pickle(
    str(MACHABEIS_CORPUS_PATH)
)

certamina_corpus = TwoLevelCorpus.from_pickle(
    str(CERTAMINA_CORPUS_PATH)
)

# Sanity checks: both uncertain works must remain holdouts from test_corpus
test_authors = set(test_corpus.get_unique_values("author"))
assert "X3" not in test_authors, "X3_machabeis must not be present in test_corpus"
assert "X4" not in test_authors, "X4_certamina must not be present in test_corpus"

certamina_authors = set(certamina_corpus.get_unique_values("author"))
assert certamina_authors == {"X4"}, (
    f"Expected certamina_corpus to contain only author X4, found: {certamina_authors}"
)

print(
    f"De Machabeis: {len(machabeis_corpus)} sentences, "
    f"{sum(len(s.text.split()) for s in machabeis_corpus)} tokens"
)

print(
    f"Certamina: {len(certamina_corpus)} sentences, "
    f"{sum(len(s.text.split()) for s in certamina_corpus)} tokens"
)

print(
    f"Test corpus: {len(test_corpus)} sentences, "
    f"{sum(len(s.text.split()) for s in test_corpus)} tokens"
)

print(
    f"Test corpus authors: "
    f"{test_corpus.get_unique_values('author')}"
)

# Analysis-only corpus used when Certamina (X4) is itself the candidate.
# X4 remains excluded from the reference pool in every other comparison.
test_plus_certamina = type(test_corpus)(
    list(test_corpus) + list(certamina_corpus),
    levels=test_corpus.levels,
)


In [ ]:
# Sampling function
def sample_corpus_to_token_count(corpus, target_tokens: int, rng: Any):
    """Sample sentences from corpus until we reach approximately target_tokens.

    Shuffles sentences and adds them until we exceed target_tokens.
    """
    sentences = list(corpus)
    total_tokens = sum(len(s.text.split()) for s in sentences)

    if total_tokens <= target_tokens:
        raise ValueError(
            f"Corpus has only {total_tokens} tokens, which is less than target {target_tokens}"
        )

    shuffled = list(sentences)
    rng.shuffle(shuffled)

    sampled = []
    current_tokens = 0
    for s in shuffled:
        sampled.append(s)
        current_tokens += len(s.text.split())
        if current_tokens >= target_tokens:
            break

    return type(corpus)(sampled, levels=corpus.levels)

In [ ]:
# Initialize Ray
ray.init(ignore_reinit_error=True)
print(f"Ray initialized with {ray.available_resources().get('CPU', 4)} CPUs")

In [ ]:
test_corpus.stats()

### De Machabeis vs Hildebert

In [ ]:
# Define the evaluation function for Ray
@ray.remote
def run_machabeis_evaluation(
    query_seed: int,
    av_seed: int,
    target_tokens: int = 750,
) -> dict:
    """Run a single AV evaluation: De Machabeis vs author H.

    query_seed controls only the random sample drawn from De Machabeis.
    av_seed controls only the internal randomness of LambdaG.
    """
    query_rng = np.random.default_rng(query_seed)
    av_rng = np.random.default_rng(av_seed)
    av = LambdaGAV()

    # Sample ~750 tokens from De Machabeis as the unknown (q_corpus)
    q_corpus = sample_corpus_to_token_count(
        machabeis_corpus,
        target_tokens,
        query_rng,
    )

    result = av.run_single_av_problem(
        known_entity="H",
        q_corpus=q_corpus,
        kr_corpus=test_corpus,
        known_tokens=2000,
        reference_tokens=20000,
        order=8,
        num_references=10,
        rng=av_rng,
    )

    return {
        "score": result["score"],
        "corrected_score_sqrt": result["corrected_score_sqrt"],
        "corrected_score_hapax": result["corrected_score_hapax"],
        "n_unknown": result["unknown_sentences_used"],
        "unknown_tokens": result["unknown_tokens_used"],
        "query_seed": query_seed,
        "av_seed": av_seed,
    }


In [ ]:
# Create 100 evaluation tasks
master_rng = np.random.default_rng(42)
TARGET_TOKENS = 750
N_EVALUATIONS = 100

query_seeds = master_rng.integers(
    0,
    2**32 - 1,
    size=N_EVALUATIONS,
    dtype=np.uint32,
)

av_seeds = master_rng.integers(
    0,
    2**32 - 1,
    size=N_EVALUATIONS,
    dtype=np.uint32,
)

print(f"Creating {N_EVALUATIONS} evaluation tasks...")
tasks = []

for i in range(N_EVALUATIONS):
    tasks.append(
        run_machabeis_evaluation.remote(
            int(query_seeds[i]),
            int(av_seeds[i]),
            TARGET_TOKENS,
        )
    )

print(f"✓ Created {len(tasks)} tasks")


In [ ]:
# Execute all tasks in parallel with progress bar
print("\nExecuting tasks in parallel...")
results = []

for _ in tqdm(range(len(tasks)), desc="Evaluations"):
    ready, _ = ray.wait(tasks, num_returns=1)
    result = ray.get(ready[0])
    results.append(result)
    tasks.remove(ready[0])

print(f"✓ Completed {len(results)} evaluations")
ray.shutdown()

In [ ]:
# Convert to DataFrame
df = pd.DataFrame(results)

print("=== Score Statistics ===")
print(df[["score", "corrected_score_sqrt", "corrected_score_hapax"]].describe())

print(f"\nUnknown token stats:")
print(f"  Mean: {df['unknown_tokens'].mean():.0f}")
print(f"  Min:  {df['unknown_tokens'].min()}")
print(f"  Max:  {df['unknown_tokens'].max()}")

In [ ]:
# Load the logistic regression model produced by the
# general hexameter calibration

model = joblib.load(MODEL_PATH)

df["probability"] = model.predict_proba(
    df["score"].values.reshape(-1, 1)
)[:, 1]

print("=== Probability Statistics ===")
print(df["probability"].describe())

print(f"\nMean probability: {df['probability'].mean():.4f}")
print(f"Median probability: {df['probability'].median():.4f}")

In [ ]:
# Visualization: histplot of raw scores + stripplot/violin of probabilities
fig, axes = plt.subplots(2, 1, figsize=(10, 7), dpi=300)

for ax in axes:
    ax.grid(False)

color = "#5f3e7f"

# Row 1: Raw score distribution with KDE overlay
ax = axes[0]
sns.histplot(data=df, x="score", bins=20, ax=ax, color=color, alpha=0.8, linewidth=0)
ax2 = ax.twinx()
sns.kdeplot(
    data=df,
    x="score",
    ax=ax2,
    color=color,
    fill=True,
    linewidth=0,
    alpha=0.25,
    bw_adjust=0.5,
)
ax2.set_yticklabels([])
ax2.set_yticks([])
ax2.set_ylabel("")
ax.set_title("Raw Score (H vs De Machabeis)", fontweight="bold", fontsize=11)
ax.set_xlabel("")
ax.set_ylabel("Count")
ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))
mean_val = df["score"].mean()
ax.axvline(
    mean_val,
    color="#e74c3c",
    linestyle="--",
    linewidth=1.5,
    label=f"Mean: {mean_val:.2f}",
)
ax.legend(loc="upper right", fontsize=9)

# Row 2: Probability stripplot + violin
ax = axes[1]
ax.grid(False)
df_plot = pd.DataFrame(
    {
        "probability": df["probability"].tolist(),
        "model": ["raw"] * len(df),
    }
)
sns.violinplot(
    data=df_plot,
    x="probability",
    y="model",
    hue="model",
    ax=ax,
    palette=[color],
    width=0.6,
    inner=None,
    density_norm="width",
    alpha=0.4,
    linewidth=0,
    legend=False,
)
sns.stripplot(
    data=df_plot,
    x="probability",
    y="model",
    hue="model",
    ax=ax,
    palette=[color],
    size=6,
    jitter=True,
    alpha=0.7,
    legend=False,
)
ax.set_title("Predicted Probability (same author)", fontweight="bold", fontsize=11)
ax.set_xlabel("P(H is author of De Machabeis)")
ax.set_ylabel("")
ax.axvline(x=0.5, color="gray", linestyle=":", alpha=0.4)

plt.tight_layout()
plt.show()

## Evaluate Against All Candidates

Run 100 paired evaluations against every author in the shared test corpus. For authors with fewer than 2000 tokens, use their total available token count as `known_size`.

The pairing is important: within replicate `i`, every candidate is evaluated on exactly the same random ~750-token sample from *De Machabeis*. The random stream used to select that sample is kept separate from the random stream used internally by LambdaG. This prevents candidate rankings from being confounded by different unknown-text samples.

*Certamina* (`X4`) is added as an **analysis-only candidate**. It remains excluded from training and calibration and is added to `kr_corpus` only when `X4` itself is the candidate; it therefore does not enter the reference population for comparisons with the other authors.


In [ ]:
# Compute known_size for each background author
author_sizes = {}

for author in sorted(test_corpus.get_unique_values("author")):
    author_corpus = test_corpus.filter_by_level("author", author)
    tokens = sum(len(s.text.split()) for s in author_corpus)
    known_size = min(2000, tokens)
    author_sizes[author] = known_size
    print(f"  {author:6s}: {tokens:6d} tokens → known_size={known_size}")

# Add Certamina (X4) as an analysis-only candidate
certamina_tokens = sum(len(s.text.split()) for s in certamina_corpus)
author_sizes["X4"] = min(2000, certamina_tokens)

print(
    f"  {'X4':6s}: {certamina_tokens:6d} tokens → "
    f"known_size={author_sizes['X4']} (Certamina; analysis-only holdout)"
)


In [ ]:
# Define the paired per-candidate evaluation function
@ray.remote
def run_evaluation_for_author(
    replicate: int,
    query_seed: int,
    av_seed: int,
    known_author: str,
    known_size: int,
    target_tokens: int = 750,
) -> dict:
    """Run one paired AV evaluation: De Machabeis vs one candidate.

    All candidates in the same replicate receive the same query_seed,
    hence exactly the same sampled portion of De Machabeis.
    av_seed is separate and candidate-specific, so LambdaG's internal
    randomness is not reused to construct the unknown sample.
    """
    import hashlib

    query_rng = np.random.default_rng(query_seed)
    av_rng = np.random.default_rng(av_seed)
    av = LambdaGAV()

    # Paired unknown sample: identical for every candidate in this replicate
    q_corpus = sample_corpus_to_token_count(
        machabeis_corpus,
        target_tokens,
        query_rng,
    )

    # Compact signature used only to verify that pairing worked correctly
    sample_text = "\n".join(s.text for s in q_corpus)
    query_signature = hashlib.sha256(
        sample_text.encode("utf-8")
    ).hexdigest()[:16]

    # Certamina (X4) is a holdout and is added only when it is itself
    # the candidate. It never enters the reference corpus used for
    # evaluations of the background authors.
    if known_author == "X4":
        kr_for_candidate = test_plus_certamina
    else:
        kr_for_candidate = test_corpus

    result = av.run_single_av_problem(
        known_entity=known_author,
        q_corpus=q_corpus,
        kr_corpus=kr_for_candidate,
        known_tokens=known_size,
        reference_tokens=20000,
        order=8,
        num_references=10,
        rng=av_rng,
    )

    return {
        "replicate": replicate,
        "author": known_author,
        "score": result["score"],
        "known_size": known_size,
        "query_seed": query_seed,
        "av_seed": av_seed,
        "query_signature": query_signature,
    }


In [ ]:
N_EVALUATIONS = 100

# Run paired evaluations for all authors
ray.init(ignore_reinit_error=True)
print(f"Ray initialized with {ray.available_resources().get('CPU', 4)} CPUs")

master_rng = np.random.default_rng(123)

# One query seed per replicate. This same seed is reused for EVERY candidate
# in that replicate, so all candidates see the identical Machabeis sample.
query_seeds = master_rng.integers(
    0,
    2**32 - 1,
    size=N_EVALUATIONS,
    dtype=np.uint32,
)

all_tasks = []

for replicate in range(N_EVALUATIONS):
    query_seed = int(query_seeds[replicate])

    for author, known_size in author_sizes.items():
        # Separate candidate-specific RNG for LambdaG internals.
        av_seed = int(master_rng.integers(0, 2**32 - 1))

        all_tasks.append(
            run_evaluation_for_author.remote(
                replicate,
                query_seed,
                av_seed,
                author,
                known_size,
                TARGET_TOKENS,
            )
        )

print(
    f"Created {len(all_tasks)} tasks for {len(author_sizes)} authors "
    f"across {N_EVALUATIONS} paired replicates"
)

# Execute all tasks
all_results = []
for _ in tqdm(range(len(all_tasks)), desc="Paired evaluations"):
    ready, _ = ray.wait(all_tasks, num_returns=1)
    result = ray.get(ready[0])
    all_results.append(result)
    all_tasks.remove(ready[0])

print(f"✓ Completed {len(all_results)} paired evaluations")
ray.shutdown()


In [ ]:
# Convert to DataFrame
# Every replicate should contain one result per candidate, all based on
# exactly the same sampled portion of De Machabeis.
df_all = pd.DataFrame(all_results)

# ---------------------------------------------------------
# Sanity checks for the paired design
# ---------------------------------------------------------
expected_authors = len(author_sizes)

signatures_per_replicate = (
    df_all.groupby("replicate")["query_signature"].nunique()
)
authors_per_replicate = (
    df_all.groupby("replicate")["author"].nunique()
)
query_seeds_per_replicate = (
    df_all.groupby("replicate")["query_seed"].nunique()
)

assert (signatures_per_replicate == 1).all(), (
    "Pairing error: at least one replicate used different Machabeis samples "
    "for different candidates."
)
assert (query_seeds_per_replicate == 1).all(), (
    "Pairing error: at least one replicate has more than one query_seed."
)
assert (authors_per_replicate == expected_authors).all(), (
    "Incomplete replicate: at least one replicate is missing candidates."
)

print(
    f"✓ Pairing check passed: {N_EVALUATIONS} replicates; "
    f"each replicate contains {expected_authors} candidates evaluated "
    "on exactly the same Machabeis sample."
)

# ---------------------------------------------------------
# Calibrated probabilities
# ---------------------------------------------------------
df_all["probability"] = model.predict_proba(
    df_all["score"].values.reshape(-1, 1)
)[:, 1]

# Summary: mean probability per author
summary = (
    df_all.groupby("author")["probability"]
    .agg(["mean", "std", "count"])
    .round(4)
    .sort_values("mean", ascending=False)
)

print("\n=== Mean Predicted Probability by Author ===")
print(summary)

# ---------------------------------------------------------
# Paired winner counts
# Which candidate has the highest probability for each
# identical Machabeis sample?
# ---------------------------------------------------------
winner_idx = df_all.groupby("replicate")["probability"].idxmax()
paired_winners = df_all.loc[
    winner_idx,
    ["replicate", "author", "probability"]
].sort_values("replicate")

winner_counts = (
    paired_winners["author"]
    .value_counts()
    .rename("wins")
    .to_frame()
)
winner_counts["win_rate"] = winner_counts["wins"] / N_EVALUATIONS

print("\n=== Paired winner counts ===")
print(winner_counts)


In [ ]:
# Optional paired diagnostics for selected candidates
# Each row compares candidates on the very same Machabeis sample.
pairwise = df_all.pivot(
    index="replicate",
    columns="author",
    values="probability",
)

selected = [a for a in ["B", "U11", "M", "HA", "U3", "FB"] if a in pairwise.columns]

if selected:
    print("=== Median paired probabilities for selected candidates ===")
    print(pairwise[selected].median().sort_values(ascending=False).round(4))

if "U11" in pairwise.columns and "B" in pairwise.columns:
    u11_over_b = (pairwise["U11"] > pairwise["B"]).mean()
    print(f"\nU11 beats B on the same Machabeis sample in {u11_over_b:.1%} of replicates")

if "U11" in pairwise.columns:
    u11_rank = pairwise.rank(axis=1, ascending=False, method="min")["U11"]
    print(f"U11 median paired rank: {u11_rank.median():.1f}")
    print(f"U11 top-1 rate: {(u11_rank == 1).mean():.1%}")
    print(f"U11 top-3 rate: {(u11_rank <= 3).mean():.1%}")


In [ ]:
# Visualization: median probability per author with IQR whiskers
import math

# Compute stats per author
stats = df_all.groupby("author")["probability"].agg(
    ["median", lambda x: x.quantile(0.25), lambda x: x.quantile(0.75)]
)
stats.columns = ["median", "q25", "q75"]
stats = stats.sort_values("median", ascending=True)
authors_ordered = stats.index.tolist()

# Dynamic figure height: more authors = taller figure
fig_height = max(8, 0.32 * len(authors_ordered))
fig, ax = plt.subplots(figsize=(12, fig_height), dpi=300)

ax.grid(False)

# Color: highlight H differently
colors = ["#e74c3c" if a == "H" else "#5f3e7f" for a in authors_ordered]

# IQR whiskers
for i, author in enumerate(authors_ordered):
    row = stats.loc[author]
    ax.plot(
        [row["q25"], row["q75"]],
        [i, i],
        color="black",
        linewidth=1,
        solid_capstyle="round",
        zorder=2,
    )
    for q in [row["q25"], row["q75"]]:
        ax.plot(q, i, "|", color="black", markersize=8, markeredgewidth=1, zorder=3)

# Median dots
ax.scatter(
    stats["median"].values,
    range(len(authors_ordered)),
    c=colors,
    s=60,
    zorder=5,
    edgecolors="black",
    linewidth=0.8,
)

ax.set_yticks(range(len(authors_ordered)))
ax.set_yticklabels(authors_ordered, fontsize=9)

# Put lowest values at the top and highest at the bottom
ax.invert_yaxis()

ax.set_title(
    "Median Predicted Probability by Candidate\n(De Machabeis, paired samples)",
    fontweight="bold",
    fontsize=13,
    pad=12,
)
ax.set_xlabel("Median P(same author)", fontsize=11)
ax.set_ylabel("Candidate", fontsize=11)

# Reference line at 0.5
ax.axvline(x=0.5, color="gray", linestyle=":", alpha=0.5)

# Leave extra room on the right for value labels
ax.set_xlim(-0.03, 1.05)
ax.set_xticks(np.arange(0, 1.01, 0.2))

# Add value labels on the right side of the plot
for i, val in enumerate(stats["median"].values):
    ax.text(
        1.04,
        i,
        f"{val:.3f}",
        va="center",
        ha="right",
        fontsize=8.5,
    )

# Improve margins
plt.subplots_adjust(left=0.18, right=0.95, top=0.92, bottom=0.07)
plt.savefig(f"Median_probability.png", dpi=300, bbox_inches="tight")
plt.show()